# Example of a pulling force

The code below generates a few frames for a spherically-shaped nucleus subject to a pulling force at the northern pole. The snapshots are generated into a subdirectory `snapshots/pull`.

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from tqdm.notebook import tqdm
import pyvista as pv
from hookeanity import System

In [ ]:
# Choose parameters and initialize the system

resolution = (10, 100) # control spatial resolution
lmax = 100 # control resolution of Sobolev-Matérn kernels
dt = 20 # time step
kbt = 4e-4 # effective temperature
gamma = 3e3
f_loc = (0.0, 0.0) # point-source force at the northern pole
f_ext = 2e-2 # magnitude of the pulling force
lam, mu = (1.4286, 0.3571) # Lamé coefficients

method = "strang" # integration method ("strang", "trotter", "euler")
seed = 0 # specify seed for the random number generator or set it to `None`
iterations = 30000 # number of time steps
nframes = 10 # desired number of frames excluding the initial one

each = iterations // nframes
assert each > 1, (
    f"number of iterations {iterations} must be greater than the number of frames {nframes}"
)
sys = System(resolution, lam, mu, dt, f_ext=f_ext, f_loc=f_loc, kbt=kbt, gamma=gamma, rng=seed, method="strang", lmax=lmax, verbose=True) 

# Prepare the snapshots directory
snapshots = Path("snapshots/pull")
snapshots.mkdir(parents=True, exist_ok=True)
for item in snapshots.iterdir():
    item.unlink()

# To accelerate snapshot generation
projection = snapshots / "projection.h5"
sys.save_projection_cache(projection, resolution[1])
print("Ready to go!")

In [ ]:
# Run the simulation
snapshot = snapshots / '0.vts'
sys.export_vts(snapshot, resolution[1], projection)
frames = [snapshot]
for it in tqdm(range(iterations), desc="Simulation"):
    sys()
    if (it + 1) % each == 0:
        snapshot = snapshots / f"{(it + 1) // each}.vts"
        sys.export_vts(snapshot, resolution[1], projection)
        frames.append(snapshot)

# Finalizing
pvd_content = (
    '<?xml version="1.0"?>\n'
    '<VTKFile type="Collection" version="0.1" byte_order="LittleEndian">\n'
    '  <Collection>\n'
)
for frame in frames:
    timestep = int(frame.stem)
    pvd_content += (
        f'    <DataSet timestep="{timestep}" '
        f'group="surface" part="0" file="{frame.name}"/>\n'
    )
pvd_content += '  </Collection>\n</VTKFile>\n'

pvd_file = snapshots / "output.pvd"
pvd_file.write_text(pvd_content)
print(f"{len(frames)} frames generated")

In [ ]:
# Visualization
reader = pv.get_reader(pvd_file)
times = reader.time_values

def get_frame(i):
    reader.set_active_time_point(i)
    mesh = reader.read()

    # Some .pvd files read as MultiBlock datasets.
    # If yours has one block per frame, this makes plotting simpler.
    if isinstance(mesh, pv.MultiBlock) and len(mesh) == 1:
        mesh = mesh[0]

    return mesh


# Compute global bounds over all frames
global_bounds = None

for i in range(len(times)):
    mesh_i = get_frame(i)
    b = mesh_i.bounds  # (xmin, xmax, ymin, ymax, zmin, zmax)

    if global_bounds is None:
        global_bounds = list(b)
    else:
        global_bounds[0] = min(global_bounds[0], b[0])
        global_bounds[1] = max(global_bounds[1], b[1])
        global_bounds[2] = min(global_bounds[2], b[2])
        global_bounds[3] = max(global_bounds[3], b[3])
        global_bounds[4] = min(global_bounds[4], b[4])
        global_bounds[5] = max(global_bounds[5], b[5])


# Pad global bounds by 25% along each axis
pad_fraction = 0.25

xmin, xmax, ymin, ymax, zmin, zmax = global_bounds

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin

# Handle nearly-flat dimensions safely
if dx == 0:
    dx = 1.0
if dy == 0:
    dy = 1.0
if dz == 0:
    dz = 1.0

global_bounds = (
    xmin - pad_fraction * dx,
    xmax + pad_fraction * dx,
    ymin - pad_fraction * dy,
    ymax + pad_fraction * dy,
    zmin - pad_fraction * dz,
    zmax + pad_fraction * dz,
)


# Initial frame
frame = 0
mesh = get_frame(frame)

plotter = pv.Plotter(notebook=True)

# Add the moving mesh and keep its actor
state = {}
state["actor"] = plotter.add_mesh(mesh, name="moving_mesh")

# Add a fixed wireframe box showing the padded overall motion bounds
box = pv.Box(bounds=global_bounds)
plotter.add_mesh(
    box,
    style="wireframe",
    opacity=0.4,
    color="black",
    name="motion_bounds",
)

plotter.add_text(
    f"Frame {frame}, time = {times[frame]}",
    name="time_label",
)

# Set the camera once using the padded global bounds
plotter.camera_position = "iso"
plotter.reset_camera(bounds=global_bounds)

plotter.show()


# Frame slider
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(times) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
)

def update_frame(change):
    frame = change["new"]
    mesh = get_frame(frame)

    # Remove only the old moving mesh.
    # Do not call plotter.clear(), because that resets the scene/camera context.
    plotter.remove_actor(state["actor"])

    # Add the new frame mesh
    state["actor"] = plotter.add_mesh(mesh, name="moving_mesh")

    # Update text label
    plotter.add_text(
        f"Frame {frame}, time = {times[frame]}",
        name="time_label",
    )

    # Keep the same camera and render
    plotter.render()


prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")

def previous_frame(_):
    frame_slider.value = max(frame_slider.min, frame_slider.value - 1)

def next_frame(_):
    frame_slider.value = min(frame_slider.max, frame_slider.value + 1)

prev_button.on_click(previous_frame)
next_button.on_click(next_frame)

display(widgets.HBox([prev_button, next_button]))

frame_slider.observe(update_frame, names="value")
display(frame_slider)